# Client Object and CLI Demo

This notebook shows how the Python client object maps to shell commands. The same question is run through the Python API and through `python -m pxfquery` commands.

In [1]:
from pathlib import Path
import os
import sys
import warnings
from IPython.display import display

warnings.filterwarnings("ignore", message="IProgress not found.*")

repo_root = Path.cwd()
if not (repo_root / "src" / "pxfquery").exists() and (repo_root.parent / "src" / "pxfquery").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

# Optional: load local environment variables for the LLM provider.
for env_file in [Path.cwd() / ".env", Path.cwd().parent / ".env", Path.home() / ".env"]:
    if env_file.exists():
        for line in env_file.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ.setdefault(key.strip(), value.strip())

from pxfquery import PxFQuery

pxf = PxFQuery()
print("PxFquery", pxf.version)
resource_status = pxf.resources.status()
resource_info = resource_status.to_dict() if hasattr(resource_status, "to_dict") else dict(resource_status)
print("Resource status:")
print({
    "available": resource_info.get("available"),
    "source": resource_info.get("source"),
    "version": resource_info.get("version"),
    "available_file_count": len(resource_info.get("available_files") or {}),
})

PxFquery 0.5.12.dev0
Resource status:
{'available': True, 'source': 'manifest', 'version': 'v20260628', 'available_file_count': 18}


In [2]:
import json
import subprocess

question = "In A549 lung cancer cells, what functional programs are changed after EGFR CRISPR knockout?"
print("Question:")
print(question)

qdata = pxf.tl.parse(question, top_n=8)
pxf.tl.answer(qdata)
answer = pxf.get.answer(qdata)

print("\nPython client answer:")
print(answer)

Question:
In A549 lung cancer cells, what functional programs are changed after EGFR CRISPR knockout?


[Parsing] start


[Parsing] done | mode=forward; context=A549 lung cancer cells; perturbation=EGFR; time=1.67s
[Matching] start


[Matching] done | matches=6; time=4.87s
[Matrix] start


[Matrix] done | profiles=6; skipped=0; time=0.41s
[Evidence] start


[Evidence] done | status=ready; time=4.61s



Python client answer:
Answer
In A549 lung cancer cells, EGFR CRISPR knockout is associated with increased interferon responses (alpha and gamma), EMT-I, and adipogenesis, and decreased cell cycle (G2/M), MYC targets, E2F targets, and G2/M checkpoint programs.

Analysis source: pxfquery 0.5.12.dev0
Evidence: inspect `answer.tables["route_summary"]` and `answer.tables["route_function_results"]`.
Figures: call `pxf.tl.figures(qdata, output_dir=...)` after `pxf.tl.answer(qdata)`.


In [3]:
print("Python object fields:")
print(list(answer.to_dict().keys()))
print("\nTables available from the Python object:")
print(list(answer.tables.keys()))
print("\nTop biological results from the Python object:")
display(answer.biological_results[:5])

Python object fields:
['question', 'interpreted_question', 'headline', 'summary', 'summary_source', 'biological_results', 'evidence', 'limitations', 'tables', 'figures', 'html', 'mcp', 'rendering_contract', 'structured_result', 'engineering']

Tables available from the Python object:
['ranked_results', 'primary_route_ranked_results', 'route_summary', 'route_target_functions', 'route_function_results', 'matrix_context', 'claim_rules']

Top biological results from the Python object:


[{'rank': 1,
  'label': 'HALLMARK_MYC_TARGETS_V2',
  'score': 5.359375,
  'direction': 'activated',
  'kind': 'activated_function',
  'source': None},
 {'rank': 2,
  'label': 'MP11 Translation initiation',
  'score': 5.35546875,
  'direction': 'activated',
  'kind': 'activated_function',
  'source': None},
 {'rank': 3,
  'label': 'MP8 Proteasomal degradation',
  'score': 5.33984375,
  'direction': 'activated',
  'kind': 'activated_function',
  'source': None},
 {'rank': 4,
  'label': 'MP25 Astrocytes',
  'score': 5.0,
  'direction': 'activated',
  'kind': 'activated_function',
  'source': None},
 {'rank': 5,
  'label': 'MP20 MYC',
  'score': 5.0,
  'direction': 'activated',
  'kind': 'activated_function',
  'source': None}]

In [4]:
def run_cli(args):
    env = os.environ.copy()
    env["PYTHONPATH"] = str(repo_root / "src") + os.pathsep + env.get("PYTHONPATH", "")
    cmd = [sys.executable, "-m", "pxfquery"] + args
    print("$", " ".join(["python", "-m", "pxfquery"] + args))
    result = subprocess.run(cmd, cwd=repo_root, env=env, text=True, capture_output=True, check=False)
    print("returncode:", result.returncode)
    if result.stdout:
        print("stdout:")
        print(result.stdout[:3000])
    if result.stderr:
        print("stderr:")
        print(result.stderr[:1200])
    if result.returncode != 0:
        raise RuntimeError("CLI command failed")
    return result

print("CLI helper ready")


CLI helper ready


In [5]:
print("Shell answer command, matching the Python client answer above:")
cli_answer = run_cli(["answer", question])

Shell answer command, matching the Python client answer above:
$ python -m pxfquery answer In A549 lung cancer cells, what functional programs are changed after EGFR CRISPR knockout?


returncode: 0
stdout:
Answer
In A549 lung cancer cells, EGFR CRISPR knockout is associated with increased interferon/immune-related programs (interferon alpha response, interferon gamma response, EMT-I, interferon/MHC-II (I), adipogenesis) and decreased cell cycle and MYC programs (cell cycle G2/M, MYC targets V2, E2F targets, G2M checkpoint, MYC).

Analysis source: pxfquery 0.5.12.dev0
Evidence: inspect `answer.tables["route_summary"]` and `answer.tables["route_function_results"]`.
Figures: call `pxf.tl.figures(qdata, output_dir=...)` after `pxf.tl.answer(qdata)`.

stderr:
[Parsing] start
[Parsing] done | mode=forward; context=A549 lung cancer cells; perturbation=EGFR; time=1.61s
[Matching] start
[Matching] done | matches=6; time=4.72s
[Matrix] start
[Matrix] done | profiles=6; skipped=0; time=0.36s
[Evidence] start
[Evidence] done | status=ready; time=5.31s



In [6]:
print("Shell parse command, corresponding to the Python intent object:")
cli_parse = run_cli(["parse", question])

Shell parse command, corresponding to the Python intent object:
$ python -m pxfquery parse In A549 lung cancer cells, what functional programs are changed after EGFR CRISPR knockout?


returncode: 0
stdout:
{
  "raw_query": "In A549 lung cancer cells, what functional programs are changed after EGFR CRISPR knockout?",
  "normalized_query": "In A549 lung cancer cells, what functional programs are changed after EGFR CRISPR knockout?",
  "query_type": "forward",
  "bio_context": "A549 lung cancer cells",
  "pert_desc": "EGFR",
  "pert_class": "genetic",
  "genetic_modality": "knockout",
  "function_desc": null,
  "activate": [],
  "suppress": [],
  "constraints": [],
  "forward_result_scope": "both",
  "top_n": null,
  "extracted_phrases": {},
  "ambiguity_flags": [],
  "missing_fields": [],
  "parse_confidence": 0.0,
  "parse_method": "llm",
  "parser_notes": [],
  "provider_evidence": {
    "provider": "deepseek",
    "base_url": "https://api.deepseek.com/v1",
    "model": "deepseek-chat",
    "started_at": "2026-06-29T15:29:34.187340+00:00",
    "finished_at": "2026-06-29T15:29:35.651313+00:00",
    "attempts": [
      {
        "attempt": 1,
        "ok": true,
     

In [7]:
print("Python intent from the same qdata:")
print(json.dumps(pxf.get.intent(qdata), indent=2)[:3000])

Python intent from the same qdata:
{
  "raw_query": "In A549 lung cancer cells, what functional programs are changed after EGFR CRISPR knockout?",
  "normalized_query": "In A549 lung cancer cells, what functional programs are changed after EGFR CRISPR knockout?",
  "query_type": "forward",
  "bio_context": "A549 lung cancer cells",
  "pert_desc": "EGFR",
  "pert_class": "genetic",
  "genetic_modality": "knockout",
  "function_desc": null,
  "activate": [],
  "suppress": [],
  "constraints": [],
  "forward_result_scope": "both",
  "top_n": null,
  "extracted_phrases": {},
  "ambiguity_flags": [],
  "missing_fields": [],
  "parse_confidence": 0.0,
  "parse_method": "llm",
  "parser_notes": [],
  "provider_evidence": {
    "provider": "deepseek",
    "base_url": "https://api.deepseek.com/v1",
    "model": "deepseek-chat",
    "started_at": "2026-06-29T15:29:09.492288+00:00",
    "finished_at": "2026-06-29T15:29:11.159615+00:00",
    "attempts": [
      {
        "attempt": 1,
        "ok"

In [8]:
print("Shell figures command, corresponding to pxf.tl.figures(qdata, ...):")
cli_figures = run_cli(["figures", question, "--output-dir", "pxfquery_cli_figures", "--format", "png"])

Shell figures command, corresponding to pxf.tl.figures(qdata, ...):
$ python -m pxfquery figures In A549 lung cancer cells, what functional programs are changed after EGFR CRISPR knockout? --output-dir pxfquery_cli_figures --format png


returncode: 0
stdout:
[
  "pxfquery_cli_figures/pxfquery_01_evidence_match_map.png",
  "pxfquery_cli_figures/pxfquery_02_function_match_heatmap.png",
  "pxfquery_cli_figures/pxfquery_03_function_consensus_bar.png"
]

stderr:
[Parsing] start
[Parsing] done | mode=forward; context=A549 lung cancer cells; perturbation=EGFR; time=1.43s
[Matching] start
[Matching] done | matches=6; time=4.35s
[Matrix] start
[Matrix] done | profiles=6; skipped=0; time=0.37s
[Evidence] start
[Evidence] done | status=ready; time=5.00s



In [9]:
print("Python save/load keeps the same query object available for later L5 output:")
path = Path("egfr_crispr_query.pkl")
pxf.tl.save(qdata, path)
restored = pxf.tl.load(path)
print("Saved query object:", path)
print("Restored answer summary preview:")
print(pxf.get.answer(restored).summary[:500])

Python save/load keeps the same query object available for later L5 output:
Saved query object: egfr_crispr_query.pkl
Restored answer summary preview:
In A549 lung cancer cells, EGFR CRISPR knockout is associated with increased interferon responses (alpha and gamma), EMT-I, and adipogenesis, and decreased cell cycle (G2/M), MYC targets, E2F targets, and G2/M checkpoint programs.
